# ChamaSmart – Model Training and Comparison

Run the cells **top to bottom, one at a time** (press Shift+Enter). Nothing needs editing unless a cell tells you so.

What this notebook does:
1. Gets your code from GitHub and your data (from the repo or Google Drive).
2. Trains **3 credit-scoring models**: Logistic Regression (baseline), Random Forest, XGBoost.
3. Trains **forecasting models**: last-value and 4-week-average baselines, ARIMA, and an LSTM.
4. Builds one comparison table for Chapter 5 and gives you the model files for the Flask services.

Every run saves its metrics, plots, random seed, library versions and git commit to Google Drive
(`MyDrive/ChamaSmart/runs/...`), so each number in your report can be traced back to a run.

**First time?** Set `QUICK_TEST = True` in step 3 for a 3-minute dry run. Set it back to `False` for the real run.

In [ ]:
# STEP 1 - Connect Google Drive and get your code from GitHub
import os, sys, subprocess

try:
    from google.colab import drive, files
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:          # lets the notebook also run on a normal computer
    IN_COLAB = False

ROOT = "/content/drive/MyDrive/ChamaSmart" if IN_COLAB else os.environ.get("CHAMA_ROOT", "./chama_outputs")
REPO_URL = "https://github.com/Juddist01/ChamaSmart"
REPO_DIR = os.environ.get("CHAMA_REPO", "/content/ChamaSmart")
os.makedirs(ROOT, exist_ok=True)

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"], check=False)
commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip() or "unknown"
print("Code version (git commit):", commit)
print("Outputs will be saved in:", ROOT)

needed = ["ml-services/credit-scoring/training/train_credit_scoring.py",
          "ml-services/forecasting/training/train_forecasting.py",
          "ml-services/forecasting/training/train_lstm_forecasting.py"]
missing = [n for n in needed if not os.path.exists(f"{REPO_DIR}/{n}")]
if missing:
    print("\nSTOP: these files are not on GitHub yet:", *missing, sep="\n  - ")
    print("Commit and push the training kit from your computer, then run this cell again.")

In [ ]:
# STEP 2 - Find the three datasets and look at them
import pandas as pd
try:
    from IPython.display import display, Image
except ImportError:
    display, Image = print, None

def find_csv(name):
    for folder in (f"{REPO_DIR}/datasets", f"{ROOT}/datasets"):
        path = f"{folder}/{name}"
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"{name} not found in the GitHub repo's datasets/ folder or in {ROOT}/datasets/. "
                            "Upload it to that Google Drive folder and run this cell again.")

CREDIT_CSV = find_csv("chama_credit_scoring_dataset.csv")
ARIMA_CSV  = find_csv("chama_forecasting_dataset.csv")
LSTM_CSV   = find_csv("chama_forecasting_lstm_dataset.csv")

for title, path in [("Credit scoring", CREDIT_CSV), ("Forecasting (ARIMA)", ARIMA_CSV), ("Forecasting (LSTM)", LSTM_CSV)]:
    d = pd.read_csv(path)
    print(f"\n{title}: {d.shape[0]} rows x {d.shape[1]} columns")
    display(d.head(3))

In [ ]:
# STEP 3 - Tell the notebook what your columns are
# The notebook guesses from your files. Read the printout, and only change a value if the guess is wrong.
QUICK_TEST = False      # True = tiny dry run to check everything works (about 3 minutes)

LABEL          = None   # credit CSV: the column that says who defaulted, e.g. "default"
POSITIVE_MEANS = "default"   # use "repayment" instead if a 1 in that column means the borrower REPAID
MEMBER         = None   # forecasting CSV: the member id column
TIME           = None   # forecasting CSV: the week/date column
VALUE          = None   # forecasting CSV: the number being forecast

credit = pd.read_csv(CREDIT_CSV)
fc = pd.read_csv(ARIMA_CSV)

def pick(cols, words, default):
    for w in words:
        for c in cols:
            if w in c.lower():
                return c
    return default

if LABEL is None:
    last = credit.columns[-1]
    LABEL = last if set(credit[last].dropna().unique()) <= {0, 1} else pick(credit.columns, ["label", "default", "target"], last)
if MEMBER is None: MEMBER = pick(fc.columns, ["member", "id"], fc.columns[0])
if TIME   is None: TIME   = pick(fc.columns, ["week", "date", "period", "time"], fc.columns[1])
if VALUE  is None:
    numeric = [c for c in fc.select_dtypes("number").columns if c not in (MEMBER, TIME)]
    VALUE = numeric[-1]

print("Credit label column :", LABEL, "| a 1 means:", POSITIVE_MEANS)
print("Forecast member col :", MEMBER)
print("Forecast time col   :", TIME)
print("Forecast value col  :", VALUE)
for name, df_, cols in [("credit", credit, [LABEL]), ("forecasting", fc, [MEMBER, TIME, VALUE])]:
    bad = [c for c in cols if c not in df_.columns]
    assert not bad, f"Column(s) {bad} not found in the {name} file. Set them by hand above."

extra = {"CHAMA_MAX_MEMBERS": "8", "CHAMA_EPOCHS": "10"} if QUICK_TEST else {}
ENV = dict(os.environ, CHAMA_ROOT=ROOT, CHAMA_REPO=REPO_DIR, CHAMA_LABEL=LABEL,
           CHAMA_POSITIVE_MEANS=POSITIVE_MEANS, CHAMA_MEMBER=MEMBER, CHAMA_TIME=TIME,
           CHAMA_VALUE=VALUE, **extra)

def run_script(relative_path, **more_env):
    """Runs one training script, prints its output live, returns the run folder it created."""
    proc = subprocess.Popen([sys.executable, f"{REPO_DIR}/{relative_path}"], env=dict(ENV, **more_env),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    run_dir = None
    for line in proc.stdout:
        if "WARNING" in line or "cuda" in line.lower():
            continue
        print(line, end="")
        if line.startswith("RUN_DIR="):
            run_dir = line.strip().split("=", 1)[1]
    if proc.wait() != 0:
        raise RuntimeError("The script failed - read the messages above.")
    return run_dir

## Credit scoring: 3 models
Splits the data 70/15/15 (train/validation/test), tunes each model on the training data only, chooses a threshold on the
validation data, and uses the test data **once** at the end.

In [ ]:
# STEP 4 - Train the credit-scoring models (Logistic Regression, Random Forest, XGBoost)
credit_dir = run_script("ml-services/credit-scoring/training/train_credit_scoring.py", CHAMA_DATA=CREDIT_CSV)
if Image:
    for name in ("roc_curve.png", "confusion_matrix_xgboost.png", "shap_summary.png"):
        p = f"{credit_dir}/{name}"
        if os.path.exists(p):
            display(Image(p))

## Forecasting: baselines, ARIMA and LSTM
ARIMA runs on two files: the 100-member file (this one produces the model your Flask service uses) and the LSTM file
(so ARIMA and LSTM are compared on exactly the same members and weeks). This is the slowest step.

In [ ]:
# STEP 5 - ARIMA (per-member) + baselines
arima_dir = run_script("ml-services/forecasting/training/train_forecasting.py",
                       CHAMA_DATA=ARIMA_CSV, CHAMA_RUN_TAG="_arima_service")
arima_lstmdata_dir = run_script("ml-services/forecasting/training/train_forecasting.py",
                                CHAMA_DATA=LSTM_CSV, CHAMA_RUN_TAG="_arima_on_lstm_data")

In [ ]:
# STEP 6 - LSTM
lstm_dir = run_script("ml-services/forecasting/training/train_lstm_forecasting.py",
                      CHAMA_DATA=LSTM_CSV, CHAMA_RUN_TAG="_lstm")
if Image:
    for name in ("training_curve.png", "sample_forecast.png"):
        display(Image(f"{lstm_dir}/{name}"))

In [ ]:
# STEP 7 - One comparison table for Chapter 5 (copy it, and screenshot it for your evidence log)
import json
from datetime import datetime

def load(path):
    return json.load(open(path))

credit_test = load(f"{credit_dir}/metrics.json")["test"]
names = {"baseline_logreg": "Logistic Regression (baseline)", "random_forest": "Random Forest", "xgboost": "XGBoost (served model)"}
credit_table = pd.DataFrame([{
    "Model": names[k], "AUC-ROC": v["auc_roc"], "F1": v["f1"], "Precision": v["precision"], "Recall": v["recall"]
} for k, v in credit_test.items()])

a = load(f"{arima_lstmdata_dir}/metrics.json")
l = load(f"{lstm_dir}/metrics.json")
if a["members_modelled"] != l["members_modelled"]:
    print("WARNING: ARIMA and LSTM used different numbers of members:", a["members_modelled"], l["members_modelled"])
forecast_table = pd.DataFrame([
    {"Method": "Last value (baseline)",      "Mean MAE": a["mean_mae_naive_last_value"]},
    {"Method": "4-week average (baseline)",  "Mean MAE": a["mean_mae_moving_average_4"]},
    {"Method": "ARIMA (per member, AIC grid)", "Mean MAE": a["mean_mae_arima"]},
    {"Method": "LSTM (shared model)",        "Mean MAE": l["mean_mae_lstm"]},
])
forecast_table["Members"] = a["members_modelled"]
forecast_table["Held-out weeks"] = a["holdout_weeks"]

print("TABLE A - Credit scoring, held-out TEST set\n")
display(credit_table)
print("\nTABLE B - Contribution forecasting, same members and same final", a["holdout_weeks"], "weeks (lower MAE is better)\n")
display(forecast_table)

summary_dir = f"{ROOT}/runs/summary"
os.makedirs(summary_dir, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
with open(f"{summary_dir}/model_comparison_{stamp}.md", "w") as f:
    f.write("## Credit scoring (test set)\n\n" + credit_table.to_markdown(index=False))
    f.write("\n\n## Forecasting (held-out weeks)\n\n" + forecast_table.to_markdown(index=False) + "\n")
    f.write(f"\nRuns: credit={os.path.basename(credit_dir)}, arima={os.path.basename(arima_lstmdata_dir)}, lstm={os.path.basename(lstm_dir)}, commit={commit}\n")
print("\nSaved to", f"{summary_dir}/model_comparison_{stamp}.md")

**Read your results honestly.** If a simple baseline beats a complex model, that is a valid finding - report it and explain why
(for example the data is synthetic and has little structure). Do not change settings after looking at the test numbers.

In [ ]:
# STEP 8 - Download the model files for your Flask services + print the version lines to pin
import shutil

def pack(run_dir, name):
    return shutil.make_archive(f"/content/{name}" if IN_COLAB else f"{ROOT}/{name}", "zip", f"{run_dir}/service_artifacts")

credit_zip = pack(credit_dir, "credit_artifacts")
forecast_zip = pack(arima_dir, "forecasting_artifacts")

cm = load(f"{credit_dir}/service_artifacts/metadata.json")["library_versions"]
fm = load(f"{arima_dir}/service_artifacts/metadata.json")["library_versions"]
print("\nPaste into ml-services/credit-scoring/requirements.txt (replace the unpinned lines):")
for lib in ("scikit-learn", "xgboost", "pandas", "numpy"):
    print(f"{lib}=={cm[lib]}")
print("\nPaste into ml-services/forecasting/requirements.txt (replace the unpinned lines):")
for lib in ("statsmodels", "pandas", "numpy"):
    print(f"{lib}=={fm[lib]}")
print("\nPython used for training:", cm["python"], "(use the same major.minor in the Dockerfiles)")

if IN_COLAB:
    files.download(credit_zip)
    files.download(forecast_zip)
else:
    print("Zips saved to", ROOT)

## Next (on your computer)
1. Unzip `credit_artifacts.zip` into `ml-services/credit-scoring/model/` and `forecasting_artifacts.zip` into `ml-services/forecasting/model/`.
2. Pin the library versions printed above in each `requirements.txt`.
3. Run `pytest -v` inside each service folder. The tests that were skipped before now run.
4. Commit the model files, then add Table A and Table B to Chapter 5 and your evidence log.